In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q timm opencv-python-headless

import os, json, random, copy, time, shutil
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from concurrent.futures import ThreadPoolExecutor
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

TEA_LEAF_ROOT = "/content/drive/MyDrive/Tea Leaf Dataset"
OUT           = "/content/drive/MyDrive/TeaLeaf_DINOv2_Study"
RESULTS_DIR   = f"{OUT}/results"
FIGURES_DIR   = f"{OUT}/figures"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

# ---- dataset -> local disk (parallel, resumable) ----
LOCAL_DATA = "/tmp/tea_leaf_local"
N_WORKERS  = 32

def _count_files(p):
    return sum(len(f) for _, _, f in os.walk(p)) if os.path.isdir(p) else 0

n_src, n_dst = _count_files(TEA_LEAF_ROOT), _count_files(LOCAL_DATA)
if n_dst < n_src:
    t0 = time.time(); tasks = []
    for root, _, files_ in os.walk(TEA_LEAF_ROOT):
        rel = os.path.relpath(root, TEA_LEAF_ROOT)
        dst_dir = LOCAL_DATA if rel == '.' else os.path.join(LOCAL_DATA, rel)
        os.makedirs(dst_dir, exist_ok=True)
        for f in files_:
            src, dst = os.path.join(root, f), os.path.join(dst_dir, f)
            if not os.path.exists(dst):
                tasks.append((src, dst))
    print(f"Copying {len(tasks)} files with {N_WORKERS} workers...")
    def _cp(t):
        try: shutil.copyfile(t[0], t[1]); return True
        except Exception: return False
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        ok = list(tqdm(ex.map(_cp, tasks), total=len(tasks), desc="copy"))
    print(f" Copied {sum(ok)}/{len(tasks)} in {time.time()-t0:.0f}s")
else:
    print(f" Local cache complete ({n_dst} files)")

classes = sorted([d for d in os.listdir(LOCAL_DATA)
                  if os.path.isdir(os.path.join(LOCAL_DATA, d))])
NUM_CLASSES     = len(classes)
CLASS_TO_IDX    = {c: i for i, c in enumerate(classes)}
healthy_classes = [c for c in classes if 'healthy' in c.lower()]
print(f"Classes ({NUM_CLASSES}): {classes}")

dv1_path = os.path.join(OUT, 'checkpoints', 'dino_vits16.pth')
if not os.path.exists(dv1_path):
    os.makedirs(os.path.dirname(dv1_path), exist_ok=True)
    !wget -q -O {dv1_path} https://dl.fbaipublicfiles.com/dino/dino_deitsmall16_pretrain/dino_deitsmall16_pretrain.pth

dino_v1 = timm.create_model('vit_small_patch16_224', pretrained=False, num_classes=0)
ckpt = torch.load(dv1_path, map_location='cpu')
sd = {k.replace('module.','').replace('backbone.','').replace('encoder.',''): v
      for k, v in ckpt.items() if 'head' not in k}
dino_v1.load_state_dict(sd, strict=False)
dino_v1.eval().to(device)
for p in dino_v1.parameters():
    p.requires_grad = False

_dv1_store = {}
dv1_last_attn = dino_v1.blocks[-1].attn

def _dv1_patched_forward(x, **kwargs):
    B, N, C = x.shape
    qkv = dv1_last_attn.qkv(x).reshape(B, N, 3, dv1_last_attn.num_heads,
                                       C // dv1_last_attn.num_heads)
    qkv = qkv.permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    scale = (C // dv1_last_attn.num_heads) ** -0.5
    cls_q = q[:, :, 0:1, :]
    cls_attn = ((cls_q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    _dv1_store['cls_attn'] = cls_attn[:, :, 0, 1:].detach().cpu()
    attn_full = ((q @ k.transpose(-2, -1)) * scale).softmax(dim=-1)
    x = (attn_full @ v).transpose(1, 2).reshape(B, N, C)
    x = dv1_last_attn.proj(x)
    x = dv1_last_attn.proj_drop(x)
    return x

dv1_last_attn.forward = _dv1_patched_forward

tf_dv1 = T.Compose([T.Resize((224,224)), T.ToTensor(),
                    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_train = T.Compose([T.Resize((224,224)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(),
                      T.RandomRotation(20), T.ColorJitter(0.2,0.2,0.2), T.ToTensor(),
                      T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
tf_test = T.Compose([T.Resize((224,224)), T.ToTensor(),
                     T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])

DV1_GRID, N_HEADS = 14, 6
topk = max(1, int(0.1 * DV1_GRID * DV1_GRID))

with torch.no_grad():
    _ = dino_v1(torch.randn(1,3,224,224).to(device))
print(f"Attention hook OK: {_dv1_store['cls_attn'].shape}")

all_images = []
for cls_name in classes:
    cls_dir = os.path.join(LOCAL_DATA, cls_name)
    for fname in sorted(os.listdir(cls_dir)):
        if fname.lower().endswith(('.jpg','.jpeg','.png')):
            all_images.append({
                'path': os.path.join(cls_dir, fname), 'class': cls_name,
                'class_idx': CLASS_TO_IDX[cls_name],
                'is_diseased': 0 if cls_name in healthy_classes else 1})

class SimpleDS(Dataset):
    def __init__(self, imgs, labels, transform):
        self.imgs, self.labels, self.transform = imgs, labels, transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        return self.transform(Image.open(self.imgs[i]['path']).convert('RGB')), self.labels[i]

class TestDS(Dataset):
    def __init__(self, imgs, transform):
        self.imgs, self.transform = imgs, transform
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        return (self.transform(Image.open(self.imgs[i]['path']).convert('RGB')),
                self.imgs[i]['is_diseased'])

n_dis = sum(x['is_diseased'] for x in all_images)
print(f"\n Ready: {len(all_images)} images | Diseased: {n_dis}, Healthy: {len(all_images)-n_dis}")

In [ ]:
import os, shutil, sys
from google.colab import files

NEEDED = ["shape_confound.py", "feature_shape_control.py",
          "few_shot_student.py", "clutter_robustness.py"]
missing = [f for f in NEEDED if not os.path.exists(f"/content/{f}")]
if missing:
    print("Select (ctrl/cmd-click):", missing)
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)

for m in NEEDED:
    sys.modules.pop(m[:-3], None)

import shape_confound as sc
import feature_shape_control as fsc
import few_shot_student as fss
import clutter_robustness as cr
print("All modules OK")

In [ ]:
import numpy as np, os
from PIL import Image
from tqdm import tqdm

d = np.load(os.path.join(RESULTS_DIR, 'crop_features.npz'), allow_pickle=True)
X, Y_bin, CLS = d['X'], d['Y'], d['CLS']
cls_names   = sorted(set(CLS.tolist()))
y_cls       = np.array([cls_names.index(c) for c in CLS])
healthy_idx = [i for i, c in enumerate(cls_names) if 'healthy' in c.lower()][0]

KEEP = []
for rec in tqdm(all_images, desc="realigning"):
    try:
        m = sc.leaf_mask(Image.open(rec['path']).convert('RGB'))
        if m is None or sc.shape_features(m) is None:
            continue
        KEEP.append(rec)
    except Exception:
        continue

assert len(KEEP) == len(Y_bin), f"misaligned: {len(KEEP)} vs {len(Y_bin)}"
print(f"aligned: {len(KEEP)} records | healthy_idx={healthy_idx}")

In [ ]:
import sys, shutil, os
from google.colab import files

if not os.path.exists("/content/few_shot_student_fast.py"):
    print("Select few_shot_student_fast.py:")
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)
sys.modules.pop("few_shot_student_fast", None)
import few_shot_student_fast as fss

res_student = fss.run(
    X=X, y_cls=y_cls, y_bin=Y_bin, KEEP=KEEP,
    class_names=cls_names, healthy_idx=healthy_idx,
    mask_fn=sc.leaf_mask, crop_fn=fsc.crop_to_leaf,
    device=device, n_shot=10, n_repeats=3,
    batch_size=128, workers=4,
    results_dir=RESULTS_DIR,
)

In [ ]:
import sys, shutil, os
from google.colab import files

if not os.path.exists("/content/clutter_robustness_fast.py"):
    print("Select clutter_robustness_fast.py:")
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)
sys.modules.pop("clutter_robustness_fast", None)
import clutter_robustness_fast as cr

res_clutter = cr.run(
    records=KEEP, model=dino_v1, tf=tf_dv1, device=device,
    mask_fn=sc.leaf_mask, crop_fn=fsc.crop_to_leaf, shape_fn=sc.shape_features,
    y_bin=Y_bin, y_cls=y_cls, class_names=cls_names, healthy_idx=healthy_idx,
    alphas=(0.0, 0.1, 0.2, 0.3, 0.4, 0.6),
    n_shot=10, n_eval=1200, n_repeats=3,
    batch=64,
    results_dir=RESULTS_DIR,
)

In [ ]:
import os, sys, shutil
import numpy as np
from PIL import Image
from tqdm import tqdm
from google.colab import files

# --- upload the modules you need ---
NEEDED = ["shape_confound.py", "feature_shape_control.py",
          "few_shot_student_fast.py", "fig_diagnostics.py"]
missing = [f for f in NEEDED if not os.path.exists(f"/content/{f}")]
if missing:
    print("Select (ctrl/cmd-click):", missing)
    up = files.upload()
    for name in up:
        dest = f"/content/{os.path.basename(name)}"
        if os.path.abspath(name) != os.path.abspath(dest):
            shutil.move(name, dest)
for m in NEEDED:
    sys.modules.pop(m[:-3], None)

import shape_confound as sc
import feature_shape_control as fsc
import few_shot_student_fast as fss
import fig_diagnostics as fd

# --- load the saved features ---
d = np.load(os.path.join(RESULTS_DIR, 'crop_features.npz'), allow_pickle=True)
X, Y_bin, CLS, SHAPE = d['X'], d['Y'], d['CLS'], d['SHAPE']
cls_names   = sorted(set(CLS.tolist()))
y_cls       = np.array([cls_names.index(c) for c in CLS])
healthy_idx = [i for i, c in enumerate(cls_names) if 'healthy' in c.lower()][0]

# --- rebuild KEEP (records aligned with X) ---
KEEP = []
for rec in tqdm(all_images, desc="realigning"):
    try:
        m = sc.leaf_mask(Image.open(rec['path']).convert('RGB'))
        if m is None or sc.shape_features(m) is None:
            continue
        KEEP.append(rec)
    except Exception:
        continue
assert len(KEEP) == len(Y_bin), f"misaligned: {len(KEEP)} vs {len(Y_bin)}"
print(f"ready: {X.shape} features | {len(KEEP)} records")

In [ ]:
# ============================================================
# CELL 3 — Diagnostic figures: confusion matrices, learning
#          curves, ROC/PR for the deployed 70-label student
#          (~8 min on GPU; rebuilds the crop cache first)
# ============================================================
import os, json

# rebuild the crop cache (/tmp is wiped between sessions) — ~3 min
crop_paths = fss.build_crop_cache(KEEP, sc.leaf_mask, fsc.crop_to_leaf)

diag = fd.build(
    X           = X,
    y_cls       = y_cls,
    y_bin       = Y_bin,
    crop_paths  = crop_paths,
    class_names = cls_names,
    healthy_idx = healthy_idx,
    device      = device,
    figdir      = FIGURES_DIR,
    n_shot      = 10,        # 10 per class = 70 labels
    epochs      = 30,
    test_frac   = 0.3,
    batch_size  = 128,
    workers     = 4,
)

with open(os.path.join(RESULTS_DIR, 'diagnostics.json'), 'w') as f:
    json.dump(diag, f, indent=2, default=float)

print("\nFigures written to", FIGURES_DIR)
for f in ['d1_confusion_binary', 'd2_confusion_7class',
          'd3_learning_curves', 'd4_roc_pr']:
    print("   ", f + ".pdf")

In [ ]:
from google.colab import files
import os

for f in ['d1_confusion_binary', 'd2_confusion_7class',
          'd3_learning_curves', 'd4_roc_pr']:
    files.download(os.path.join(FIGURES_DIR, f + '.png'))